In [1]:
import pandas as pd
import numpy as np
import re
from rapidfuzz import fuzz, process
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
from spellchecker import SpellChecker
import os

import warnings
warnings.filterwarnings("ignore")

os.getcwd()

'c:\\Users\\Admin\\Desktop\\P&P Round 18pt3\\Google sku mapping'

In [2]:

corrections_df = pd.read_excel("SKU_mapping/sku_mapping_dictionaries.xlsx",sheet_name="corrections")
high_freq_series_df = pd.read_excel("SKU_mapping/sku_mapping_dictionaries.xlsx",sheet_name="high_freq_series_names")
unit_map_df = pd.read_excel("SKU_mapping/sku_mapping_dictionaries.xlsx",sheet_name="unit_map")
unit_list_df = pd.read_excel("SKU_mapping/sku_mapping_dictionaries.xlsx",sheet_name="unit_list")
series_list_df = pd.read_excel("SKU_mapping/sku_mapping_dictionaries.xlsx",sheet_name="series_list")
product_keywords_df = pd.read_excel("SKU_mapping/Keywords.xlsx",sheet_name="Product Final")
brand_keywords_df = pd.read_excel("SKU_mapping/Keywords.xlsx",sheet_name="Brand Final")
brand_dict_df = pd.read_excel("SKU_mapping/product_brand_dict.xlsx",sheet_name="Brand Name Mapping")
color_list_df = pd.read_excel("SKU_mapping/colour_mapping_dict.xlsx",sheet_name="Colour List")
color_mapping_df =  pd.read_excel("SKU_mapping/colour_mapping_dict.xlsx",sheet_name="Colour Dictionary")
product_brand_mapping_df = pd.read_excel("SKU_mapping/product_brand_dict.xlsx",sheet_name="Product Brand Mapping")
short_form_dict_df = pd.read_excel("SKU_mapping/short_form_dictionary.xlsx")
brand_supplier_df = pd.read_excel("SKU_mapping/sku_mapping_dictionaries.xlsx",sheet_name="brand_supplier_list")
brand_var_df =  pd.read_excel("SKU_mapping/sku_mapping_dictionaries.xlsx",sheet_name="brands_var_list")
avg_price_df = pd.read_excel("Paints CSV Report - May'25.xlsx",sheet_name='Sheet2')
size_df = pd.read_excel("SKU_mapping/Paint - SKU Master - 06-01-2026.xlsx",sheet_name='FInal SKu')
size_df = size_df.drop(columns = ['Period','Comment','Price Segment'])

def split_and_strip(input_string):
    parts = input_string.split('-')
    if re.search(r'\d', parts[-1]):
        parts.pop() 
    result = '-'.join(parts)
    return result

def check_if_unit(sku_nm):
    sku = str(sku_nm)
    match = bool(re.search(r'(\d*\.?\d+)',sku[-1:]))
    return match    

#size_df['missing_unit'] = size_df[["Droid SKU New Name Jan'25",'Pack Size']].apply(lambda x: check_if_unit(x[1]) if 'Tools for Paints' not in x[0] else False,axis=1)
#size_df['Pack Size'] = size_df[['Pack Size','missing_unit']].apply(lambda x: str(x[0]) +' '+ 'L' if x[1] == True else x[0],axis = 1)

master_file = size_df.drop(columns = 'Base Name')
master_file = master_file.rename(columns = {"Droid SKU New Name Dec'25":'SKU Name','Company Name':'Brand Name','Brand Name':'Series Name','Category - MP':'Product'})
master_file['SKU Name'] = master_file["SKU Name"].apply(split_and_strip)
#master_file = master_file.drop(columns = 'missing_unit')
master_file['Series Name'] = master_file['Series Name'].str.upper()
master_file['Brand Name'] = master_file['Brand Name'].replace({'Asian Paints':'asianpaints'})
master_file['Brand Name'] = master_file[['SKU Name','Brand Name']].apply(lambda x: 'British Paints' if 'British Paints' in x[0] else x[1],axis=1)

master_df = master_file[['SKU Name','Pack Size']]

size_df = size_df[["Droid SKU New Name Dec'25","Pack Size","Company Name"]]
size_df = size_df[size_df['Company Name'] != 'Other Brands']
size_df = size_df.rename(columns = {"Droid SKU New Name Dec'25":'SKU Name with packsize'})
size_df['SKU Name'] = size_df['SKU Name with packsize'].apply(lambda x: split_and_strip(x) if 'Tools for Paints' not in x else x) ### New added
size_df['Final Name'] = size_df['SKU Name with packsize']
#size_df['Final Name'] = size_df[['SKU Name','Pack Size']].apply(lambda x: str(x[0]) + '-' + str(x[1]) if 'Tools for Paints' not in x[0] else x[0],axis = 1) ### New added   column used in master list instead of SKU Name with Packsize
size_df['Pack Size'] =size_df['Pack Size'].astype(str)
size_df['unit_only'] = size_df['Pack Size'].str.replace(r'[^a-zA-Z]', '',regex=True)
size_df['packsize_only'] = size_df['Pack Size'].str.extract(r'(\d*\.?\d+)')


brand_keywords_df["Keyword"] = brand_keywords_df["Keyword"].str.upper()
brand_keywords_df["Brand"] = brand_keywords_df["Brand"].str.upper()

product_keywords_df["Keyword"] = product_keywords_df["Keyword"].str.upper()
product_keywords_df["Brand"] = product_keywords_df["Product"].str.upper()

units_list = unit_list_df["Units"].to_list()
unit_map = (unit_map_df.set_index("Unit").to_dict())["Standardized Unit"]
high_freq_series_names = high_freq_series_df["Series"].tolist()
corrections = (corrections_df.set_index("Expression").to_dict())["Correction"]
product_keywords = product_keywords_df.groupby("Product")['Keyword'].unique().to_dict()
brand_keywords = brand_keywords_df.groupby("Brand")['Keyword'].unique().to_dict()
brand_dict = brand_dict_df.groupby("Brand")['Keyword'].unique().to_dict()
color_list = color_list_df["Colours"].to_list()
color_mapping = (color_mapping_df.set_index("Short Form").to_dict())["Colour"]
product_brand_mapping = product_brand_mapping_df.groupby("Product")["Brand"].unique().to_dict()
short_form_dict = (short_form_dict_df.set_index("Short Form").to_dict())["Expansion"]
brand_supplier = brand_supplier_df["Brands"].to_list()
brands_var_lst = brand_var_df["Brands"].to_list()
series_list = series_list_df['Series'].to_list()



output_file_processed_description = 'SKU_mapping/processed_product_descriptions.xlsx'
high_freq_output_file = 'SKU_mapping/untagged_high_frequency_words.xlsx'
supplier_re = "SKU_mapping/Supplier_RE.xlsx"
not_paints_keyword=list(pd.read_excel("SKU_mapping/Keywords.xlsx",sheet_name="not_paint")["Not Paint Keyword"])
common_keywords = pd.read_excel("SKU_mapping/Keywords.xlsx",sheet_name="common_keywords")
ocr_output = pd.read_csv('output_highest_confidence_all_files.csv')  # to be updated 
input_df = ocr_output[['S. No.','description','supplier_name']]


input_df = input_df.rename(columns = {'description':'Description'})
#input_df = input_df.iloc[0:10]
input_df.to_excel('Rawsku.xlsx',index=False)
input_file_skuraw ='Rawsku.xlsx'

latest_inshop_file="FormData-R16 pt1.csv"  #to be updated



In [3]:
product_brand_reverse = {}

for product_name, brands in product_brand_mapping.items():

    for brand in brands:

        if product_name not in product_brand_reverse:

            product_brand_reverse[product_name] = []

        product_brand_reverse[product_name].append(brand)



# Sort units by length descending so longer units (LTR, ML) match before short ones (L, G) - avoids partial matches
units_sorted = sorted(units_list, key=len, reverse=True)
# Pack size: allow at start, or after space/hyphen/underscore - e.g. "4 L", "enamel_4 L", "product-4L"
# Between number and unit: optional space, hyphen, underscore, or slash - e.g. 4L, 4 L, 4-L, 4_L, 4/L
pack_size_units_pattern = r'(?:^|(?<=[\s\-_]))[\(]*\s*(\d+(?:\.\d+)?(?:\s*[xX*]\s*\d+(?:\.\d+)?)?)(?:\s*[-/_]?\s*)(' + '|'.join(re.escape(u) for u in units_sorted) + r')\b'
 

def parse_product_description(description):
    # Initialize the fields
    brand = 'Unknown'
    series_name = 'Unknown'
    product_name = 'Unknown'
    color = 'Unknown'
    pack_size = 'Unknown'
    units = 'Unknown'
    remaining_description = ''

    # Handle non-string descriptions
    if pd.isna(description):
        description = ''
    elif not isinstance(description, str):
        description = str(description)

    description = description.lower()
   
    token = word_tokenize(description)
    token_series=pd.Series(token)
    token_series=token_series.replace(short_form_dict)
    token=list(token_series)
    description=" ".join(token)
    
    # Convert to uppercase for uniformity
    desc = description.upper()

    # Remove non-ASCII characters to handle OCR errors
    desc = re.sub(r'[^\x00-\x7F]+', ' ', desc)

    # Apply corrections
    for pattern, replacement in corrections.items():
        desc = re.sub(pattern, replacement, desc)

    # Remove extra spaces
    desc = re.sub(r'\s+', ' ', desc).strip()

    # Map color abbreviations to full names
    for abbr, full_name in color_mapping.items():
        desc = re.sub(r'\b' + re.escape(abbr) + r'\b', full_name, desc)

    # Extract Pack Size and Units - use LAST match (pack size typically at end of description)
    # to avoid capturing product codes like AC21G, AC2G which have number+letter patterns
    pack_size_matches = list(re.finditer(pack_size_units_pattern, desc))
    pack_size_match = pack_size_matches[-1] if pack_size_matches else None
    if pack_size_match:
        pack_size = pack_size_match.group(1).replace(' ', '')
        units = pack_size_match.group(2)
        desc = desc.replace(pack_size_match.group(0), '').strip()

    # Extract Color
    for color_option in color_list:
        if re.search(r'\b' + re.escape(color_option.upper()) + r'\b', desc):
            color = color_option
            if color_option != 'WHITE':
                desc = re.sub(r'\b' + re.escape(color_option.upper()) + r'\b', '', desc).strip()
            break

    common_keywords = pd.read_excel("SKU_mapping/Keywords.xlsx",sheet_name="common_keywords")
    key_word_list = common_keywords["Words"].tolist()

    # Extract Series Name and Brand using the series_brand_mapping
    series_found = False
    for brand_key, series_list in brand_keywords.items():
        for series in series_list:
            # Use word boundaries to prevent partial matches
            if re.search(r'\b' + re.escape(series.upper()) + r'\b', desc,flags=re.IGNORECASE):
                series_name = series
                brand = brand_key
                if series_name.lower() not in key_word_list:
                  desc = re.sub(r'\b' + re.escape(series.upper()) + r'\b', '', desc).strip()
                  series_found = True
                  break
        if series_found:
            break
        excluded_words = ['Opus','OPUS','opus','White','white','WHITE','HIL','Hil','hil']
        pattern_birla = rf'\bBirla\b(?!\s+({"|".join(excluded_words)}))'
    
        if re.search(pattern_birla,desc,flags=re.IGNORECASE):
            brand = "Birla HIL_Birla Opus_Birla White"

    # If brand not found via series, extract Brand using the brand_dict
    if brand == 'Unknown':
        brand_found = False
        for full_name, abbreviations in brand_dict.items():
            for abbr in abbreviations:
                # Use word boundaries for accurate matching
                if re.search(r'\b' + re.escape(abbr.upper()) + r'\b', desc,flags=re.IGNORECASE):
                    brand = full_name
                    if brand.lower() not in key_word_list:
                      desc = re.sub(r'\b' + re.escape(abbr.upper()) + r'\b', '', desc).strip()
                      brand_found = True
                      break
            if brand_found:
                break

    # Extract Product Name using keywords
    product_name_found = False
    for product, keywords in product_keywords.items():
        for keyword in keywords:
            if re.search(r'\b' + re.escape(keyword.upper()) + r'\b', desc,flags=re.IGNORECASE):
                product_name = product
                desc = re.sub(r'\b' + re.escape(keyword.upper()) + r'\b', '', desc,flags=re.IGNORECASE).strip()
                product_name_found = True
                break
        if product_name_found:
            break

    # Assign Brand based on Product if not found earlier
    if brand == 'Unknown' and product_name != 'Unknown':
        possible_brands = product_brand_reverse.get(product_name, [])
        if len(possible_brands) == 1:
            brand = possible_brands[0]
        elif len(possible_brands) > 1:
            # Use fuzzy matching to find the best brand match
            brand_match = process.extractOne(desc, possible_brands, scorer=fuzz.token_sort_ratio)
            if brand_match and brand_match[1] > 85:
                brand = brand_match[0]

    # Remove color words from description
#    for color_option in color_list:
#        if color_option != 'White':
#            desc = re.sub(r'\b' + re.escape(color_option.upper()) + r'\b', '', desc).strip()

    # Remove any extra spaces
    remaining_description = ' '.join(desc.split())

    # Return the parsed components
    return {
        'Brand': brand,
        'Series Name': series_name,
        'Product': product_name,
        'Color': color,
        'Pack Size': pack_size,
        'Units': units,
        'Remaining Description': remaining_description
    }
 

def process_excel_file(input_df,output_file, high_freq_output_file):

    # Read the Excel file

    try:

        df = pd.read_excel(input_df)
        df.rename(columns={"description":"Description"},inplace=True)

    except Exception as e:

        print(f"Error reading Excel file: {e}")

        return

 

    # Check if 'Description' column exists

    if 'Description' not in df.columns:

        print("The Excel file must contain a 'Description' column.")

        return

 

    # Convert the 'Description' column to strings and handle NaNs

    df['Description'] = df['Description'].astype(str)

    df['Description'] = df['Description'].fillna('')

 

    # Extract high-frequency words

    untagged_words = extract_high_frequency_words(df['Description'])

 

    # Save untagged high-frequency words to a file

    try:

        untagged_words.to_excel(high_freq_output_file, index=False)

        print(f"Untagged high-frequency words have been saved to '{high_freq_output_file}'.")

    except Exception as e:

        print(f"Error writing untagged words to Excel file: {e}")

 

    # Apply the parsing function to each product description

    parsed_data = df['Description'].apply(parse_product_description)
    
 

    # Create a DataFrame from the parsed data

    parsed_df = pd.DataFrame(parsed_data.tolist())

 

    # Concatenate the original DataFrame with the parsed DataFrame

    result_df = pd.concat([df, parsed_df], axis=1)

    
   

    # Save the result to a new Excel file

    try:

        result_df.to_excel(output_file, index=False)

        print(f"Processed data has been saved to '{output_file}'.")

    except Exception as e:

        print(f"Error writing to Excel file: {e}")

 

def extract_high_frequency_words(descriptions, frequency_threshold=10):

    # Combine all descriptions into one large string

    combined_text = ' '.join(descriptions).upper()

    # Remove non-ASCII characters
    combined_text = re.sub(r'[^\x00-\x7F]+', ' ', combined_text)

    # Remove special characters and digits

    combined_text = re.sub(r'[^\w\s]', ' ', combined_text)

    combined_text = re.sub(r'\d+', ' ', combined_text)

 

    # Split into words

    words = combined_text.split()

 

    # Count word frequencies

    word_counts = Counter(words)

 

    # Filter words above the frequency threshold

    high_freq_words = {word: count for word, count in word_counts.items() if count >= frequency_threshold}

 

    # Collect all tagged words from parsing dictionaries

    tagged_words = collect_tagged_words()

 

    # Identify untagged high-frequency words

    untagged_words = {word: count for word, count in high_freq_words.items() if word not in tagged_words}

 

    # Convert to DataFrame for output

    untagged_words_df = pd.DataFrame(list(untagged_words.items()), columns=['Word', 'Frequency'])

    untagged_words_df = untagged_words_df.sort_values(by='Frequency', ascending=False)

 

    return untagged_words_df

 

def collect_tagged_words():

    # Collect words from corrections

    corrections_words = set()

    for pattern, replacement in corrections.items():

        # Remove regex patterns to extract words


        word = re.sub(r'[\\\^\$\.\|\?\*\+\(\)\[\]\{\}]', '', pattern)

        word = word.replace('\\b', '')

        word = word.replace('\\', '')

        word = word.strip()

        if word:

            corrections_words.add(word.upper())

 

    # Collect words from brand_dict

    brand_words = set()

    for brand, abbreviations in brand_dict.items():

        brand_words.add(brand.upper())

        for abbr in abbreviations:

            brand_words.add(abbr.upper())

 

    # Collect words from product_keywords

    product_words = set()

    for product, keywords in product_keywords.items():

        product_words.add(product.upper())

        for keyword in keywords:

            product_words.add(keyword.upper())

 

    # Collect words from color_list and color_mapping

    color_words = set(color.upper() for color in color_list)

    for abbr, full_name in color_mapping.items():

        color_words.add(abbr.upper())

        color_words.add(full_name.upper())

 

    # Collect words from series_brand_mapping

    series_words = set()

    for brand, series_list in brand_keywords.items():

        for series in series_list:

            series_words.add(series.upper())

 

    # Combine all tagged words

    tagged_words = corrections_words.union(brand_words, product_words, color_words, series_words)

    return tagged_words

        



process_excel_file(input_file_skuraw,output_file_processed_description, high_freq_output_file)



Untagged high-frequency words have been saved to 'SKU_mapping/untagged_high_frequency_words.xlsx'.
Processed data has been saved to 'SKU_mapping/processed_product_descriptions.xlsx'.


In [4]:
df = pd.read_excel(output_file_processed_description)
known_df=df[df['Brand']!="Unknown"]
unknown_df=df[df['Brand']=="Unknown"]

re_df=pd.read_excel(supplier_re)
re_dict=re_df.groupby('Supplier Name RE')['Brand'].unique().to_dict()

def get_brand(supplier_nm):
    fin_supplier = supplier_nm
    if type(supplier_nm)==str:
        for br in re_dict.keys():
            if br in supplier_nm.lower():
                if len(re_dict[br])>1:
                    fin_supplier=re_dict[br][0]+"_"+re_dict[br][1]+"_"+re_dict[br][2]
                else:
                    fin_supplier=re_dict[br][0]
                break
    return fin_supplier

unknown_df['supplier_name_clean']=unknown_df['supplier_name'].apply(get_brand)
known_df['supplier_name_clean'] = known_df['supplier_name'].apply(get_brand)

known_df['supplier_name_clean'] = known_df[['supplier_name','supplier_name_clean']].apply(lambda x:'British Paints' if 'british paints division' in str(x[0]).lower() else x[1],axis=1)
unknown_df['supplier_name_clean'] = unknown_df[['supplier_name','supplier_name_clean']].apply(lambda x: 'British Paints' if 'british paints division' in str(x[0]).lower() else x[1],axis=1)


unknown_df['Brand']=unknown_df[['supplier_name_clean','Brand']].apply(lambda z:z[0] if (z[0] in brand_supplier) else 'Unknown',axis = 1)

df = pd.concat([known_df,unknown_df])

rem_df=df[df['Brand']!="Birla HIL_Birla Opus_Birla White"]
split_df=df[df['Brand']=="Birla HIL_Birla Opus_Birla White"]
birla_lst=[]
for rpt in range(3):
    temp_df=split_df.copy()
    temp_df['Brand']="Birla HIL_Birla Opus_Birla White".split("_")[rpt]
    birla_lst.append(temp_df)
concat_df=pd.concat(birla_lst)
brand_df=pd.concat([rem_df,concat_df])

###
rem_df=brand_df[brand_df['Brand']!="JK Cement_JK MAXX Paints_JK Lakshmi"]
split_df=brand_df[brand_df['Brand']=="JK Cement_JK MAXX Paints_JK Lakshmi"]
jk_lst=[]
for rpt in range(3):
    temp_df=split_df.copy()
    temp_df['Brand']="JK Cement_JK MAXX Paints_JK Lakshmi".split("_")[rpt]
    jk_lst.append(temp_df)
concat_df=pd.concat(jk_lst)
brand_df=pd.concat([rem_df,concat_df])
###

def product_check(pr_name):
    flag=0
    if type(pr_name)==str:
        for kw in not_paints_keyword:
            if kw in pr_name.lower():
                flag= 1
                break
    return flag
    
brand_df['Not Paint Product Flag']=brand_df['Description'].apply(lambda z:product_check(z))

paints_df=brand_df[brand_df['Not Paint Product Flag']==0]

#%%R

paints_df['Brand']=paints_df['Brand'].replace({"Unknown":"Other Brands","OTHER BRANDS":"Other Brands"}) 

all_brands_df= pd.concat([paints_df,known_df])

brand_supplier_ls_refined=list(set(brand_supplier)-set(["Birla HIL_Birla Opus_Birla White",'JK Cement_JK MAXX Paints_JK Lakshmi']))
all_brands_df['Brand']=all_brands_df[['supplier_name_clean','Brand']].apply(lambda z:z[0] if (z[0]!=z[1] and z[0] in brand_supplier_ls_refined and z[1] != 'British Paints') else z[1],axis=1)


all_brands_df.to_csv("SKU_mapping/Corrected File.csv",index=False)


df=all_brands_df

df.drop_duplicates(inplace=True)
df.dropna(subset=['Description'],inplace=True)


df["Brand"] = df["Brand"].str.title()
df['Brand'] = df['Brand'].replace({"Asian Paints": "asianpaints","Jsw":"JSW","Jk Cement":"JK Cement","Asianpaints":"asianpaints",
                                   "Dr. Fixit":"DR. FIXIT",'Birla Hil':"Birla HIL","Jk Maxx Paints":"JK MAXX Paints",'Jk Cement ':'JK Cement','Mrf':'MRF'})

#%%

db_df= master_file
db_df.dropna(subset=['SKU Name'],inplace=True)


def sku_refiner(row):
    sku=row['SKU Name']
    
    brand=row['Brand Name']
    sku_cleaned=sku.replace("-","")
    sku_cleaned=sku_cleaned.replace(brand,"")
    sku_cleaned=sku_cleaned.strip()
    sku_cleaned=sku_cleaned.lower()
    sku_cleaned=re.sub(" +"," ",sku_cleaned)
    sku_cleaned=sku_cleaned.replace("(cw tinting base)","")

    return sku_cleaned

db_df['SKU refined']=db_df[['SKU Name','Brand Name']].apply(sku_refiner,axis=1)
db_df=db_df[['Brand Name','SKU Name','SKU refined','Product', 'Series Name']].drop_duplicates()
db_df.rename(columns={"Brand Name":"Brand"},inplace=True)

#%%

df['Desc_cleaned']=df['Description'].apply(lambda z:z.lower())
df['Desc_cleaned']=df['Desc_cleaned'].apply(lambda z:re.sub("\d"," ",z))
df['Desc_cleaned']=df['Desc_cleaned'].apply(lambda z:re.sub("[^a-z\s]"," ",z))
df['Desc_cleaned']=df['Desc_cleaned'].apply(lambda z:re.sub(" +"," ",z))

stop_words = list(stopwords.words('english'))
stop_words.extend(['ml','ltr','lt','kg','l',"buy","for","price","litre","at","from",
                   "india","best","prices","online","get",'gst','hi','cgst','bag','gms',
                   'central','state','ap'])

txt=list(df['Desc_cleaned'])
txt=" ".join(txt)

text_tokens=word_tokenize(txt)
text_tokens=[tkn for tkn in text_tokens if tkn not in stop_words]
fq=nltk.probability.FreqDist(t for t in text_tokens)
zz=fq.most_common(50)

def cleaner(txt_str):
    tokens=word_tokenize(txt_str)
    tokens=[t for t in tokens if t not in stop_words]
    
    tokens_series=pd.Series(tokens)
    tokens_series=tokens_series.replace(short_form_dict)
    tokens=list(tokens_series)
    tokens=[z for z in tokens if len(z)>=2]

    
    cleaned_str=" ".join(tokens)
    fin_str=cleaned_str
    for b in brands_var_lst:
        if cleaned_str.startswith(b):
#            fin_str=cleaned_str.replace(b,"",count=1)
            fin_str=re.sub(b,"",cleaned_str,count=1)

    fin_str=fin_str.strip()        
    return fin_str

df['Desc_cleaned']=df['Desc_cleaned'].apply(cleaner)

df=df[df['Desc_cleaned']!=""]

# string matching with fuzzywuzzy

def matcher_brand_specific_all(string,brand,product,series):
    if brand!="Unknown":
        brand_df=db_df[db_df['Brand']==brand]
        if series in series_list:
            brand_df = brand_df[brand_df['Series Name'].str.contains(series)]
        if product!="Unknown":
            if product=="Emulsion":
                brand_df=brand_df[(brand_df['Product']=="Interior Emulsion") | (brand_df['Product']=="Exterior Emulsion")]
            elif product == "Putty":
                brand_df=brand_df[(brand_df['Product']=="Cement Putty") | (brand_df['Product']=="Acrylic Putty")]
            else:
                brand_df=brand_df[brand_df['Product']==product]
            
    elif brand=='Unknown':
        brand_df=db_df
        if product!="Unknown":
            if product=="Emulsion":
                brand_df=brand_df[(brand_df['Product']=="Interior Emulsion") | (brand_df['Product']=="Exterior Emulsion")]
            elif product == "Putty":
                brand_df=brand_df[(brand_df['Product']=="Cement Putty") | (brand_df['Product']=="Acrylic Putty")]
            else:
                brand_df=brand_df[brand_df['Product']==product]
            
    choices=list(brand_df['SKU refined'].unique()) 
    match=process.extractOne(string,choices,scorer=fuzz.token_sort_ratio)
    try:
        match_name=match[0]
        match_score=match[1]
        return (match_name,match_score)
    except:
        return(-1,-1)

#df['Brand'] = df['Brand'].replace({'asianpaints':'Asian Paints'})

df[['SKU refined','Match Score']]=df[['Desc_cleaned','Brand','Product','Series Name']].apply(lambda s:matcher_brand_specific_all(s[0],s[1],s[2],s[3]),axis=1,result_type='expand')

db_df_unique = db_df.drop_duplicates(subset=['SKU Name'])


df_merged = pd.merge(df, db_df_unique[['SKU refined', 'SKU Name','Brand']], on=['SKU refined','Brand'], how='left')

df_merged=df_merged.sort_values("Match Score",ascending=False).drop_duplicates(subset=['S. No.'],keep='first')

df_merged['Brand'] = df_merged['Brand'].replace({'British Paints':"Berger"})


df_merged.to_csv("SKU_mapping/master_matching.csv",index=False)


In [5]:
def normalize_and_convert_units(value, unit):

    if pd.isna(value) or pd.isna(unit):

        return None, 'Unknown'

   
    unit = unit.strip().upper()
    # Treat standalone M as ML (e.g., 100M -> 100 ML)
   
    unit = unit_map.get(unit, unit)  # fallback method

 

    try:

        val = float(str(value).strip())

    except:

        val = None

 

    if val is None or unit == 'Unknown':

        return None, 'Unknown'

 

    if unit == 'ML':

        val = val / 1000.0

        unit = 'L'

    elif unit == 'G':

        val = val / 1000.0

        unit = 'KG'

 

    return val, unit

 

def parse_pack_size(pack_size):

    if pd.isna(pack_size) or not isinstance(pack_size, str):

        return None, 'Unknown'

   

    parts = pack_size.strip().split()

    if len(parts) == 2:

        val_str, unit_str = parts

        try:

            val = float(val_str)

        except:

            val = None

        return val, unit_str

    else:

        match = re.match(r"^(\d+(\.\d+)?)(.*)$", pack_size.strip())

        if match:

            val = float(match.group(1))

            unit_str = match.group(3).strip()

            return val, unit_str if unit_str else 'Unknown'

       

    return None, 'Unknown'

 

def load_and_prepare_data(master_df, source_path):

    #master_df = pd.read_excel(master_path)

    source_df = pd.read_csv(source_path)

    master_df[['Raw_Pack_Num', 'Raw_Unit']] = master_df.apply(lambda r: pd.Series(parse_pack_size(r['Pack Size'])), axis=1)
    master_df[['Pack Size Num', 'Norm Units']] = master_df.apply(lambda r: pd.Series(normalize_and_convert_units(r['Raw_Pack_Num'], r['Raw_Unit'])), axis=1)
    master_df['SKU_lower'] = master_df['SKU Name'].str.lower().str.strip()


    source_df[['Pack Size Num', 'Norm Units']] = source_df.apply(lambda r: pd.Series(normalize_and_convert_units(r['Pack Size'], r['Units'])), axis=1)

    source_df['SKU_lower'] = source_df['SKU Name'].str.lower().str.strip()

 

    master_df['Pack_Full'] = master_df.apply(lambda r: f"{r['Pack Size Num']:.6g} {r['Norm Units']}"

                                             if r['Pack Size Num'] is not None and r['Norm Units']!='Unknown' else "Unknown", axis=1)

    source_df['Pack_Full'] = source_df.apply(lambda r: f"{r['Pack Size Num']:.6g} {r['Norm Units']}"

                                             if r['Pack Size Num'] is not None and r['Norm Units']!='Unknown' else "Unknown", axis=1)

   

    return master_df, source_df

 

def fuzzy_match_sku_and_pack(source_row, master_df):

    if source_row['S. No.']%100==0:

        print(source_row['S. No.'])

    source_sku = source_row['SKU_lower']

    pack_num = source_row['Pack Size Num']

    norm_unit = source_row['Norm Units']

 

    all_skus = master_df['SKU_lower'].unique()

    best_match_sku = None

    if len(all_skus) > 0 and type(source_sku)!=float:

        result = process.extractOne(source_sku, all_skus, scorer=fuzz.WRatio)

        if result:

            best_sku, score, _ = result

#            best_sku, score= result

 

            if score >= 60:
                

                best_match_sku = best_sku

 

    if best_match_sku is None:

        return None, None

 

    candidate_skus = master_df[master_df['SKU_lower'] == best_match_sku].copy()

 

    if candidate_skus.empty:

        return None, None

 

    best_match = None

    if pack_num is not None and not candidate_skus['Pack Size Num'].isna().all():

        candidate_skus = candidate_skus.dropna(subset=['Pack Size Num'])

        if candidate_skus.empty:

            if not candidate_skus.empty:

                best_match = candidate_skus.iloc[0]

        else:

            candidate_skus['diff'] = (candidate_skus['Pack Size Num'] - pack_num).abs()

           

            if candidate_skus['diff'].isna().all():

                best_match = candidate_skus.iloc[0]

            else:

                best_idx = candidate_skus['diff'].idxmin()

                best_match = candidate_skus.loc[best_idx]

    else:

        # pack_num is None - return SKU but leave pack blank
        best_match = candidate_skus.iloc[0]
        return best_match['SKU Name'], None

 

    if best_match is not None:

        return best_match['SKU Name'], best_match['Pack_Full']

    else:

        return None, None
    


 

def match_source_to_master(master_df, source_df):

    merged = pd.merge(source_df, master_df, left_on=['SKU_lower','Pack_Full'],

                      right_on=['SKU_lower','Pack_Full'], how='left', suffixes=('', '_master'))

    if 'Matched SKU' not in merged.columns:

        merged['Matched SKU'] = None

    if 'Matched Pack_Full' not in merged.columns:

        merged['Matched Pack_Full'] = None

    no_match = merged[merged['SKU Name_master'].isna()].copy()
    no_match.to_csv('Unmatched Packsizes.csv',index=False)
    print(len(no_match))

 

    if not no_match.empty:

        no_match[['Matched SKU','Matched Pack_Full']] = no_match.apply(

            lambda r: pd.Series(fuzzy_match_sku_and_pack(r, master_df)), axis=1

        )

        merged.update(no_match[['Matched SKU','Matched Pack_Full']])

 

    merged['Final SKU Name'] = merged['Matched SKU'].fillna(merged['SKU Name_master'])

    merged['Final Pack Size'] = merged['Matched Pack_Full'].fillna(merged['Pack_Full'])

 

    final_df = merged[['S. No.', 'SKU Name', 'Pack Size', 'Units', 'Final SKU Name', 'Final Pack Size']]


   

    return final_df

   

master_df, source_df = load_and_prepare_data(master_df, "SKU_mapping/master_matching.csv")

final_result = match_source_to_master(master_df, source_df)

# Do NOT use Pack Size/Units from master_matching as fallback. Use only match result.
# If original Pack Size is Unknown, Final Pack Size must also be Unknown
def _choose_pack_size(row):
    orig = row['Pack Size']
    if orig is None or pd.isna(orig) or str(orig).strip() in ('', 'Unknown', 'nan', 'NaN', 'None'):
        return 'Unknown'
    final = row['Final Pack Size']
    if final is None or pd.isna(final) or str(final).strip() in ('', 'Unknown', 'nan', 'NaN', 'None'):
        return 'Unknown'
    return final
final_result["Final Pack Size"] = final_result.apply(_choose_pack_size, axis=1)

final_result["Units"] = final_result["Units"].replace(unit_map)
                                               
final_result.to_excel("SKU_mapping/PS_matched_output.xlsx", index=False)





df = ocr_output.copy()

# Initialize spell checker
spell = SpellChecker()

# Add custom words to the spell checker's dictionary
custom_words = [
    'lakh', 'lakhs', 'crore', 'crores', 'rupees', 'paise', 'only', 'inr', 'rs', 'lac', 'paise',
    'hundred', 'thousand', 'million', 'billion', 'trillion'
]
spell.word_frequency.load_words(custom_words)

# Number mapping
number_dict = {
    # Units and Teens
    'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
    'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10,
    'eleven': 11, 'twelve': 12, 'thirteen': 13, 'fourteen': 14,
    'fifteen': 15, 'sixteen': 16, 'seventeen': 17, 'eighteen': 18,
    'nineteen': 19,

    # Tens
    'twenty': 20, 'thirty': 30, 'forty': 40, 'fifty': 50,
    'sixty': 60, 'seventy': 70, 'eighty': 80, 'ninety': 90,

    # Scales
    'hundred': 100, 'thousand': 1000, 'lakh': 100000, 'lac': 100000,
    'crore': 10000000,

    # Common misspellings and typos
    'for': 4, 'fou': 4, 'our': 4,  # 'our' for 'four'
    'ive': 5, 'fiv': 5, 'iv': 4,
    'ight': 8, 'eigth': 8, 'aight': 8,  # 'ight' for 'eight'
    'to': 2, 'too': 2,
    'sax': 6, 'saxteen': 16,  # 'sax' for 'six'
    'sinaty': 60, 'sinty': 60, 'sventy': 70, 'sunding': 70, 'susting': 60, 'susty': 60,  # Typos for 'sixty' and 'seventy'
    'ninty': 90, 'ninaty': 90,
    'thite': 30, 'thirty': 30,
    'lath': 100000, 'laith': 100000, 'lad': 100000,  # Typos for 'lakh'
    'thousnad': 1000, 'housand': 1000, 'tousand': 1000, 'thosand': 1000,  # Typos for 'thousand'
    'ixteen': 16, 'suteen': 16, 'suteen': 16,  # Typos for 'sixteen'
    'sateen': 17, 'suteen': 16, 'sunding': 70,  # Typos for 'seventeen' and 'sixteen'
    'se': 6, 'tree': 3, 'conty': 20, 'conty': 20, 'sainting': 60, 'tour': 4,
}

# Preprocessing functions
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s\-]', ' ', text)
    return text

def replace_misspellings(text):
    # Replace common misspellings
    misspellings = {
        'our': 'four',
        'ive': 'five',
        'ight': 'eight',
        'laith': 'lakh',
        'lad': 'lakh',
        'sax': 'six',
        'ixteen': 'sixteen',
        'sinty': 'sixty',
        'sinaty': 'sixty',
        'sventy': 'seventy',
        'sunding': 'seventy',
        'ninty': 'ninety',
        'fity': 'fifty',
        'thite': 'thirty',
        'suteen': 'sixteen',
        'sateen': 'seventeen',
        'for': 'four',
        'to': 'two',
        'tousand': 'thousand',
        'thosand': 'thousand',
        'housand': 'thousand',
        'lath': 'lakh',
        'tree': 'three',
        'se': 'six',
        'sainting': 'sixty',
        'conty': 'twenty',
        'susting': 'sixty',
        'susty': 'sixty',
        'tour': 'four',
        'conty': 'twenty',
        'saxteen': 'sixteen',
        'light': 'eight',
        'singh': 'six',
        'singhteen': 'sixteen',
    }
    tokens = text.split()
    corrected_tokens = [misspellings.get(word, word) for word in tokens]
    return ' '.join(corrected_tokens)

def correct_spelling(text):
    tokens = text.split()
    corrected_tokens = []
    for word in tokens:
        corrected = spell.correction(word)
        if corrected is None or corrected == word:
            corrected_tokens.append(word)
        else:
            corrected_tokens.append(corrected)
    return ' '.join(corrected_tokens)

def words_to_num(text):
    tokens = text.replace('-', ' ').split()
    total = 0
    current = 0
    for word in tokens:
        if word in ['and', 'only', 'rupees', 'rs', 'inr', 'paise', 'rupee']:
            continue
        elif word in number_dict:
            scale = number_dict[word]
            if scale == 100:
                if current == 0:
                    current = 1
                current *= scale
            elif scale in [1000, 100000, 10000000]:
                if current == 0:
                    current = 1
                current *= scale
                total += current
                current = 0
            else:
                current += scale
        else:
            # Handle unknown words or log them
            pass
    total += current
    return total

def process_amount(row):
    text = row['Total_Amount_in_words']
    text = normalize_text(text)
    text = replace_misspellings(text)
    text = correct_spelling(text)
    amount = words_to_num(text)
    return amount

# Apply the processing to each row
df['Total_Amount_in_words_converted'] = df.apply(process_amount, axis=1)
#os.chdir("..")
os.makedirs("post_processing files",exist_ok=True)

# Save the results
df.to_excel('post_processing files//output_highest_confidence_all_files_words_corrected.xlsx', index=False)


#Correct numbers
def parse_number(text):
    text = str(text).strip()
    if not text:
        return None

    # Split text on newline characters and take the first part
    text = text.split('\n')[0]

    # Remove any non-digit, non-comma, non-period characters except for spaces
    text = re.sub(r'[^\d.,\s]', '', text)
    
    # Replace spaces between digits with a period (assuming it's a decimal point)
    text = re.sub(r'(?<=\d) +(?=\d)', '.', text)
    
    # Remove any remaining spaces
    text = text.replace(' ', '')
    
    # Identify decimal separator (match decimal separator followed by digits at the end)
    match = re.search(r'([.,])(\d+)$', text)
    if match:
        decimal_sep = match.group(1)
        decimal_digits = match.group(2)
        integer_part = text[:match.start(1)]
        
        # Determine which separator is the decimal separator
        if decimal_sep == ',':
            # Comma is decimal separator, remove periods as thousand separators
            integer_part = integer_part.replace('.', '')
            decimal_point = '.'
        else:
            # Period is decimal separator, remove commas as thousand separators
            integer_part = integer_part.replace(',', '')
            decimal_point = '.'
        
        # Remove any remaining thousand separators from integer part
        integer_part = integer_part.replace(',', '').replace('.', '')
        
        # Construct the standardized number string
        number_str = integer_part + decimal_point + decimal_digits
    else:
        # No decimal separator found
        # Remove all commas and periods from the integer part
        number_str = text.replace(',', '').replace('.', '')
    
    try:
        return float(number_str)
    except ValueError:
        return None


# List of columns to apply the parse_number function to
columns_to_parse = [
    'total_amount', 
    'Total_Taxable_Amount', 
    'unit_price', 
    'quantity', 
    'Taxable-Amount', 
    'Value', 
    'Volume', 
    'amount'
]

# Apply parsing function to each column
for column in columns_to_parse:
    df[f'{column}_in_Numbers'] = df[column].apply(lambda x: parse_number(str(x)))

# Save the results back to an Excel file
output_file = 'post_processing files//output_highest_confidence_all_files_words_corrected_numbers_converted.xlsx'  # Replace with your desired output file path
df.to_excel(output_file, index=False)

print("Parsing completed. Results saved to", output_file)


# Define the acceptable percentage tolerances
tolerance_percentage = 30
additional_tolerance = 20  # Tolerance for the new logic

# Renaming columns:

df.rename(columns={'Taxable-Amount_in_Numbers':'Taxable Amount', 'Value_in_Numbers':'Amt_w/o_disc_tax', 'amount_in_Numbers':'Amt',
                   'unit_price_in_Numbers':'Price','quantity_in_Numbers':'Qty'},inplace=True)

# Filter out rows where all three key metrics are missing
df = df.dropna(subset=['Taxable Amount', 'Amt_w/o_disc_tax', 'Amt'], how='all')  # At least one of them must be present

# Define a helper function to calculate if a value lies within tolerance
def is_within_tolerance(base, target, tolerance):
    if pd.notna(base) and pd.notna(target) and base > 0:  # Avoid division by zero and missing values
        deviation = abs((target - base) / base) * 100
        return deviation <= tolerance
    return False

# Add a column to flag if Amt is within tolerance of either Taxable Amount or Amt_w/o_disc_tax
df['Amt_Correct'] = (
    df.apply(lambda row: 
             is_within_tolerance(row['Taxable Amount'], row['Amt'], tolerance_percentage) or
             is_within_tolerance(row['Amt_w/o_disc_tax'], row['Amt'], tolerance_percentage), axis=1)
)

# Add a column to flag potential errors in Price or Qty
df['Potential_Error_in'] = df.apply( lambda row: 'Price or Qty' if row['Amt_Correct'] else 'Amt, Taxable Amount, or Amt_w/o_disc_tax', axis=1)

# === Original Green_Code Calculation ===
def is_green_code(row):
    if pd.notna(row['Price']) and pd.notna(row['Qty']) and pd.notna(row['Amt']):
        # 30% tolerance as per the code logic
        return abs((row['Price'] * row['Qty']) - row['Amt']) <= 0.30 * (row['Price'] * row['Qty'])
    return False

# Add Green_Code_Before column
df['Green_Code_Before'] = df.apply(is_green_code, axis=1)
# === End of Original Green_Code Calculation ===

# Define a helper function to check if two values deviate by less than a specified percentage
def within_additional_tolerance(val1, val2, tolerance):
    if pd.notna(val1) and pd.notna(val2) and val1 > 0:
        deviation = abs((val1 - val2) / val1) * 100
        return deviation <= tolerance
    return False

# Function to apply the new Amt assignment logic
def assign_amt(row):
    # Check if Green_Code_Before is False and Amt is missing
    if not row['Green_Code_Before'] and pd.isna(row['Amt']):
        taxable = row['Taxable Amount']
        amt_wo_disc = row['Amt_w/o_disc_tax']
        # Check if both Taxable Amount and Amt_w/o_disc_tax are present
        if pd.notna(taxable) and pd.notna(amt_wo_disc):
            # Check if they deviate by less than the additional tolerance
            if within_additional_tolerance(taxable, amt_wo_disc, additional_tolerance):
                # Assign the greater of the two as Amt
                new_amt = max(taxable, amt_wo_disc)
                return new_amt
    # If conditions are not met, return the original Amt
    return row['Amt']

# Apply the Amt assignment
df['Amt'] = df.apply(assign_amt, axis=1)

# Flag rows where Amt was updated
df['Amt_Updated'] = df.apply(lambda row: 'Amt updated based on Taxable Amount and Amt_w/o_disc_tax' 
                if (not row['Green_Code_Before'] and pd.isna(row['Amt']) and 
                    pd.notna(row['Taxable Amount']) and pd.notna(row['Amt_w/o_disc_tax']) and 
                    within_additional_tolerance(row['Taxable Amount'], row['Amt_w/o_disc_tax'], additional_tolerance)) 
                else '', axis=1)

# === Recalculate Green_Code After Amt Update ===
df['Green_Code_After'] = df.apply(is_green_code, axis=1)
# === End of New Logic ===

# Save back to Excel
output_file = 'post_processing files//output_with_amt_and_green_code.xlsx'
df.to_excel(output_file, index=False)


tax_wo_disc_tolerance = 20  # 20% deviation between Taxable Amount and Amt_w/o_disc_tax
amt_deviation_tolerance = 30  # 30% deviation of Amt from Taxable Amount and Amt_w/o_disc_tax

# ============================
# Step 3: Implement Correction Logic
# ============================

# Define a helper function to calculate percentage deviation
def calculate_percentage_deviation(base, target):
    """
    Calculate the percentage deviation between base and target.
    """
    if pd.notna(base) and pd.notna(target) and base != 0:
        return abs((target - base) / base) * 100
    return np.nan

# Calculate deviations
df['Deviation_Amt_Taxable'] = df.apply(lambda row: calculate_percentage_deviation(row['Taxable Amount'], row['Amt']), axis=1)

df['Deviation_Amt_Wo_Disc'] = df.apply(lambda row: calculate_percentage_deviation(row['Amt_w/o_disc_tax'], row['Amt']), axis=1)

df['Deviation_PriceQty_Taxable'] = df.apply(lambda row: calculate_percentage_deviation(row['Taxable Amount'], row['Price'] * row['Qty']) if pd.notna(row['Price']) and pd.notna(row['Qty']) else np.nan,axis=1)

df['Deviation_PriceQty_Amt_Wo_Disc'] = df.apply(lambda row: calculate_percentage_deviation(row['Amt_w/o_disc_tax'], row['Price'] * row['Qty']) if pd.notna(row['Price']) and pd.notna(row['Qty']) else np.nan,axis=1)

# Define the correction condition
# Condition 1: Either Taxable Amount or Amt_w/o_disc_tax deviates by >=30% from Amt
condition_deviation = (((df['Deviation_Amt_Taxable'] >= amt_deviation_tolerance) & df['Taxable Amount'].notna()) |  ((df['Deviation_Amt_Wo_Disc'] >= amt_deviation_tolerance) & df['Amt_w/o_disc_tax'].notna()))

# Condition 2: Price*Qty is within 20% deviation of Taxable Amount or Amt_w/o_disc_tax
condition_price_qty = (    ((df['Deviation_PriceQty_Taxable'] <= tax_wo_disc_tolerance) & df['Taxable Amount'].notna()) |  ((df['Deviation_PriceQty_Amt_Wo_Disc'] <= tax_wo_disc_tolerance) & df['Amt_w/o_disc_tax'].notna()))

# Combined condition
combined_condition = condition_deviation & condition_price_qty

# Additionally, ensure that both Taxable Amount and Amt_w/o_disc_tax are within 20% of Price*Qty if both are present
# This part is a bit unclear, but based on user description, we need to assign higher if both are within 20% of Price*Qty
# So we check if both deviations are <=20%
both_within_tolerance = (    (df['Deviation_PriceQty_Taxable'] <= tax_wo_disc_tolerance) &    (df['Deviation_PriceQty_Amt_Wo_Disc'] <= tax_wo_disc_tolerance))

# Final condition: combined_condition and (both_within_tolerance if both are present)
final_condition = combined_condition & (    (~df['Taxable Amount'].isna() & ~df['Amt_w/o_disc_tax'].isna() & both_within_tolerance) |  (df['Taxable Amount'].isna() | df['Amt_w/o_disc_tax'].isna()))
    
# Explanation:
# - If both 'Taxable Amount' and 'Amt_w/o_disc_tax' are present, then both should be within tolerance of Price*Qty
# - If only one is present, only that one needs to be within tolerance

# Apply the correction: assign the higher of Taxable Amount and Amt_w/o_disc_tax to Amt
# Define a new column 'Amt_Corrected' initially as the original 'Amt'
df['Amt_Corrected'] = df['Amt']

# Rows that meet the final_condition will have 'Amt' corrected
df.loc[final_condition, 'Amt_Corrected'] = df.loc[final_condition, ['Taxable Amount', 'Amt_w/o_disc_tax']].max(axis=1)

# ============================
# Step 4: Recalculate Green_Code_After
# ============================

# Define a function to recalculate Green_Code_After based on corrected Amt
def recalculate_green_code(row, tolerance=0.30):
    """
    Recalculate Green_Code_After based on the corrected Amt.
    Green_Code is True if |Price*Qty - Amt_Corrected| <= 30% of Price*Qty
    """
    if pd.notna(row['Price']) and pd.notna(row['Qty']) and pd.notna(row['Amt_Corrected']):
        expected_amt = row['Price'] * row['Qty']
        deviation = abs(expected_amt - row['Amt_Corrected'])
        tolerance_value = tolerance * expected_amt
        return deviation <= tolerance_value
    return False

# Apply the recalculation
df['Green_Code_After_Corrected'] = df.apply(recalculate_green_code, axis=1)

# ============================
# Step 5: Flag Corrections
# ============================

# Create a flag to indicate corrections
df['Amt_Corrected_Flag'] = np.where(    final_condition,    'Amt corrected to higher of Taxable Amount and Amt_w/o_disc_tax',  '')

# ============================
# Step 6: Clean Up and Save
# ============================

# Optionally, drop the intermediate deviation columns
df.drop([  'Deviation_Amt_Taxable',
    'Deviation_Amt_Wo_Disc',
    'Deviation_PriceQty_Taxable',
    'Deviation_PriceQty_Amt_Wo_Disc'], axis=1, inplace=True)

# Save the corrected DataFrame to a new Excel file
output_file = 'post_processing files//output_with_amt_and_green_code_output_corrected_final.xlsx'
df.to_excel(output_file, index=False)

# ============================
# Optional: Preview Corrected Entries
# ============================

# Select corrected entries
corrected_entries = df[final_condition]

"""
print("Corrected Entries:")
print(corrected_entries[[
    'description', 'Price', 'Qty', 'Taxable Amount',
    'Amt_w/o_disc_tax', 'Amt', 'Amt_Corrected',
    'Green_Code_After', 'Green_Code_After_Corrected',
    'Amt_Corrected_Flag'
]])
"""

print("\nAll corrections have been applied and saved to:", output_file)


output_csv_path="Corrections complete.csv"

def extract_x1_x2(box):
    if pd.isna(box):
        return None, None
    try:
        coords = re.findall(r"\(([^)]+)\)", box)
        points = [tuple(map(float, coord.split(','))) for coord in coords]
        
        # Ensure there are at least 2 points
        if len(points) < 2:
            raise ValueError("Insufficient points in bounding box")
        
        x1 = points[0][0]
        x2 = points[1][0]
        return x1, x2
    except Exception as e:
        print(f"Error processing bounding box '{box}': {e}")
        return None, None

# Apply the function safely
df[['x1', 'x2']] = df['quantity_bounding_box'].apply(
    lambda box: pd.Series(extract_x1_x2(box))
)

df['width'] = df['x2'] - df['x1']

def compare_widths_and_create_column(df):
    # Iterate over each unique filename
    for filename in df['file_name'].unique():
        # Extract the rows where green_code_after is True
        true_rows = df[(df['file_name'] == filename) & (df['Green_Code_After_Corrected'] == True)]
        
        # If there are no rows with green_code_after True, skip this filename
        if true_rows.empty:
            continue
        
        # Get the width of the first 'green_code_after' True entry (assuming one entry)
        true_width = true_rows['width'].iloc[0]
        
        # Create the 'quantity_box_big' column and initialize as False
        df.loc[(df['file_name'] == filename) & (df['Green_Code_After_Corrected'] == False), 'quantity_box_wider'] = df.loc[
            (df['file_name'] == filename) & (df['Green_Code_After_Corrected'] == False), 'width'] > true_width
    
    # Fill NaN values in 'quantity_box_big' (for True rows) with False
    df['quantity_box_wider'] = df['quantity_box_wider'].fillna(False)
    
    return df

# Apply the function
df = compare_widths_and_create_column(df)

df['quantity'] = pd.to_numeric(df['Qty'], errors='coerce')  # Coerce errors will turn invalid values to NaN
df['quantity'] = df['quantity'].fillna(0).astype(int)  # Fill NaN values with 0 and convert to integer


def generate_substrings(number):
    num_str = str(number)
    substrings = []
    for i in range(len(num_str)):
        for j in range(i + 1, len(num_str) + 1):
            substrings.append(num_str[i:j])
    return substrings

# Function to add rendition columns based on the 'quantity' column
def add_rendition_columns(df, column_name):
    # Generate substrings for each row in the specified column
    all_renditions = df[column_name].apply(generate_substrings)
    
    # Determine the maximum number of substrings to create columns dynamically
    max_renditions = max(all_renditions.apply(len))
    
    # Add new columns for each rendition
    for i in range(max_renditions):
        df[f'rendition {i + 1}'] = all_renditions.apply(lambda x: x[i] if i < len(x) else None)
    return df

# Apply the function to add rendition columns without modifying other columns
df = add_rendition_columns(df, 'quantity')

# Function to check for the correct rendition based on the unit price and amount
def find_correct_quantity(df, unit_price_col, amount_col):
    # Iterate through rendition columns to find the correct quantity
    rendition_columns = [col for col in df.columns if col.startswith('rendition')]
    correct_quantities = []

    for _, row in df.iterrows():
        correct_quantity = row['Qty']
        for col in rendition_columns:
            rendition = row[col]
            if rendition is not None:
                try:
                    rendition = int(rendition)  # Convert rendition to integer
                    calculated_amount = rendition * int(float(row[unit_price_col]))
                    lower_bound = int(float(row[amount_col])) * 0.75
                    upper_bound = int(float(row[amount_col])) * 1.25
                    # Check if the calculated amount is within the threshold
                    if lower_bound <= calculated_amount <= upper_bound:
                        correct_quantity = rendition
                        break
                except (ValueError, TypeError):  # Handle cases where conversion fails
                    pass
        correct_quantities.append(correct_quantity)
    
    # Add a new column to store the correct quantities
    df['correct quantity'] = correct_quantities
    return df

# Apply the function to find the correct quantity
df = find_correct_quantity(df, 'Price', 'Amt')


def decimal_corrector(row):
    try:
        price=int(row['Price'])
        qty=int(row['correct quantity'])
        amt=int(row['Amt_Corrected'])
        price_decimal_before=str(price)[:len(str(price))-2]
        price_decimal_after=str(price)[-2:]
        rectified_price=float(price_decimal_before+"."+price_decimal_after)
        price_calculated_amount = rectified_price * qty
        price_lower_bound = amt * 0.75
        price_upper_bound = amt * 1.25
        
        amt_decimal_before=str(amt)[:len(str(amt))-2]
        amt_decimal_after=str(amt)[-2:]
        rectified_amt=float(amt_decimal_before+"."+amt_decimal_after)
        calculated_amount = price * qty
        amt_lower_bound = calculated_amount * 0.75
        amt_upper_bound = calculated_amount * 1.25
        # Check if the calculated amount is within the threshold
        if price_lower_bound <= price_calculated_amount <= price_upper_bound:
            print(row['S. No.'])
            return (rectified_price,qty,amt)

        # Check if the calculated amount is within the threshold
        elif amt_lower_bound <= rectified_amt <= amt_upper_bound:
            print(2)
            return (price,qty,rectified_amt)

        else:
            return(price,qty,amt)
    except:
            return("NA","NA","NA")

df[['Fin_Price','Fin_Qty','Fin_Amt']]=df.apply(decimal_corrector,axis=1,result_type='expand')

df.to_csv(output_csv_path,index=False)


275
200
300
400
Parsing completed. Results saved to post_processing files//output_highest_confidence_all_files_words_corrected_numbers_converted.xlsx

All corrections have been applied and saved to: post_processing files//output_with_amt_and_green_code_output_corrected_final.xlsx


In [6]:

calc_df=pd.read_csv("Corrections complete.csv")
#calc_df.rename(columns={"Unnamed: 0":"S. No."},inplace=True)
calc_df=calc_df[['S. No.','file_name','Fin_Price','Fin_Qty','Fin_Amt']]
calc_df['Fin_Price']=calc_df['Fin_Price'].replace({0:"NA"})
calc_df['Fin_Qty']=calc_df['Fin_Qty'].replace({0:"NA"})
calc_df['Fin_Amt']=calc_df['Fin_Amt'].replace({0:"NA"})
calc_df=calc_df.replace({"NA":np.nan})
calc_df.dropna(inplace=True)
calc_df['Metrics inline flag']=calc_df.apply(lambda z:1 if abs((z['Fin_Price']*z['Fin_Qty'])-z['Fin_Amt'])/z['Fin_Amt']<0.3 else 0,axis=1)
calc_df=calc_df[calc_df['Metrics inline flag']==1]
#calc_df['Result ID']=calc_df['file_name'].apply(lambda x:x.split('_')[7])
#calc_df=calc_df.merge(inshop_df_format)
calc_df['City'] = calc_df['file_name'].apply(lambda x:x.split('_')[3])
calc_df['Outlet Name'] = calc_df['file_name'].apply(lambda x:x.split('_')[4])
calc_df['Outlet Code'] = calc_df['file_name'].apply(lambda x:x.split('_')[5])


sku_df=pd.read_csv("SKU_mapping/master_matching.csv")
sku_df=sku_df[['S. No.','SKU Name']]

sku_master_df=master_file

# Build sku_combined and get Final Pack Size from PS_matched_output (from Cell 4)
sku_combined=sku_df.merge(sku_master_df[['SKU Name','Brand Name','Product']].drop_duplicates())
ps_matched = pd.read_excel("SKU_mapping/PS_matched_output.xlsx")
sku_combined = sku_combined.merge(ps_matched[['S. No.', 'Final Pack Size', 'Final SKU Name']], on='S. No.', how='left')
sku_combined['Final Pack Size'] = sku_combined['Final Pack Size'].fillna('Unknown').replace({np.nan: 'Unknown'})
sku_combined['Final Pack Size'] = sku_combined['Final Pack Size'].apply(lambda x: 'Unknown' if pd.isna(x) or str(x).strip() in ('', 'nan', 'Unknown', 'None') else x)

combined_df=calc_df.merge(sku_combined,how='inner')
# Use Final Pack Size from PS_matched_output to build Final SKU Name
combined_df['Final SKU Name'] = combined_df.apply(
    lambda s: f"{s['SKU Name']}-{s['Final Pack Size']}"
              if str(s['Final Pack Size']) not in ('Unknown', 'nan', '') and pd.notna(s['Final Pack Size'])
              else str(s['SKU Name']),
    axis=1
)

combined_df.to_csv("Intermediate file.csv",index=False)
#exp_df=combined_df[['City', 'Outlet Code', 'Outlet Name','Fin_Price', 'Fin_Qty', 'Fin_Amt','Final SKU Name','Brand Name', 'Product']]
combined_df.drop(columns=['SKU Name'],inplace=True)
#source_df = exp_df
source_df=combined_df

def process_string(s):
    parts = s.split('-')
    return '-'.join(parts) if len(parts) == 2 else '-'.join(parts[:-1])

avg_price_df["SKU Name"] = avg_price_df["SKU"].apply(process_string)
avg_price_df['Full Pack Size'] = avg_price_df["SKU"].apply(lambda x: x.split('-')[-1])
avg_price_df = avg_price_df[~avg_price_df['Full Pack Size'].apply(lambda x: not bool(re.search(r'\d',x.split('-')[-1])))]
avg_price_df['avg_price'] = avg_price_df.groupby('SKU Name')['Price'].transform('mean')
avg_price_df = avg_price_df.drop_duplicates(subset=['SKU Name'],keep='first')

# subset_df = rows where Final Pack Size is Unknown (need find_packsize); exclude Other Brands
subset_df = source_df[(source_df['Final Pack Size'] == 'Unknown') & (source_df['Brand Name'] != 'Other Brands')]
# rest_df = all other rows (Final Pack Size known from PS_matched, or Other Brands)
rest_df = source_df[~source_df.index.isin(subset_df.index)]

def convert_int(x):
    if isinstance(x,float) and x.is_integer():
        return int(x)
    return x

def find_closest_match(num,num_list):
    return min(num_list, key = lambda x: abs(float(x)-float(num)))

def find_packsize(row,size_df,avg_price_df):
    row_df = pd.DataFrame([row],columns=subset_df.columns)
    compare_df = row_df.merge(avg_price_df,how='inner',left_on='Final SKU Name',right_on='SKU Name')
    if compare_df['avg_price'].empty:
        return 'Unknown'
    else:
        sku_name = str(compare_df['SKU Name'].iloc[0])
        master_subset = size_df[size_df['SKU Name with packsize'].str.startswith(sku_name)]
        size_options = master_subset['packsize_only'].to_list()
        size_options = [float(num) for num in size_options]
        fin_price = row['Fin_Price']
        avg_price = float(compare_df['avg_price'].iloc[0])
        est_packsize = float(fin_price)/float(avg_price)
        matched_packsize = find_closest_match(est_packsize,size_options)
        matched_packsize = convert_int(matched_packsize)
        unit = master_subset['unit_only'].iloc[0]
        final_packsize = str(matched_packsize)+' '+str(unit)
        return final_packsize


subset_df['Final Pack Size'] = subset_df.apply(lambda x:find_packsize(x,size_df,avg_price_df),axis=1)

subset_df['Final SKU Name'] = subset_df[['Final SKU Name','Final Pack Size']].apply(lambda x: x[0] +'-'+ x[1] if x[1] != 'Unknown' else x[0],axis = 1)

# rest_df already has Final Pack Size from PS_matched_output merge - do not overwrite

final_result = pd.concat([subset_df,rest_df])

master_list = list(size_df['Final Name'])


final_result = final_result.merge(size_df[['SKU Name with packsize']],how='left',right_on='SKU Name with packsize',left_on='Final SKU Name')

nan_final_result=final_result[final_result['SKU Name with packsize'].isnull()]
#assert (nan_final_result['Final Pack Size'] == 'Unknown').sum()<20
nan_final_result = nan_final_result[nan_final_result['Final Pack Size'] != 'Unknown']
                             
nan_final_result = nan_final_result[nan_final_result['Brand Name'] !='Other Brands']

others_df = final_result[final_result['Brand Name'] == 'Other Brands']
others_df = others_df[others_df['SKU Name with packsize'].isnull()]
known_final_result=final_result[final_result['SKU Name with packsize'].notnull()]

#known_final_result = pd.concat([others_df,known_final_result])

nan_final_result['SKU Name'] = nan_final_result['Final SKU Name'].apply(process_string)

def convert_to_int(value):
    try:
        fin_val=int(float(value))
    except :
        fin_val=value
    return fin_val


def check_packsize_match(row,master_list):
    orig_sku_name=str(row['SKU Name'])
    orig_pack_size=str(row['Final Pack Size'])

   
    sku_name_processed=None
    sku_name_processed_1=orig_sku_name+"-"+orig_pack_size
    if sku_name_processed_1 in master_list:
        return sku_name_processed_1
     
    unit_only=re.sub(r'[^a-zA-Z]', '',orig_pack_size)
    packsize_only=re.sub(r'[^0-9.]', '',orig_pack_size)

    packsize_only_1=float(packsize_only) * 1000 if float(packsize_only) < 1 else packsize_only
    packsize_only_1 = convert_to_int(packsize_only_1)
    unit_only_1='ML' if unit_only == 'L' else unit_only
    packsize_full = str(packsize_only_1)+unit_only_1 if 'K' in unit_only else str(packsize_only_1)+" "+unit_only_1
    sku_name_processed_2 = orig_sku_name+"-"+packsize_full
    if sku_name_processed_2 in master_list:
        return sku_name_processed_2
  
    packsize_full_1 = '1' if 'Tools for Paints' in orig_sku_name else packsize_full
    sku_name_processed_3 = orig_sku_name+"-"+packsize_full_1
    if sku_name_processed_3 in master_list:
        return sku_name_processed_3
    
    unit_only_2 = unit_only.title()
    packsize_full_2 = str(packsize_only_1)+" " + unit_only_2
    sku_name_processed_4 = orig_sku_name+"-"+packsize_full_2
    if sku_name_processed_4 in master_list:
        return sku_name_processed_4
   
    unit_only_3 = 'GM' if 'Pidicrete URP' in orig_sku_name and float(packsize_only) < 100 else unit_only_2
    packsize_full_3 = str(packsize_only_1) + " " + unit_only_3
    sku_name_processed_5 = orig_sku_name if 'Roff' in orig_sku_name or 'Nerolac - Tile Adhesives' in orig_sku_name or 'British Paints' in orig_sku_name else orig_sku_name+"-"+packsize_full_3
    if sku_name_processed_5 in master_list:
        return sku_name_processed_5
    
    packsize_full_4 = packsize_only+'0'+' '+unit_only
    sku_name_processed_6 = orig_sku_name +'-'+ packsize_full_4
    if sku_name_processed_6 in master_list:
        return sku_name_processed_6
    
    sku_name_processed_7 = orig_sku_name +'-'+packsize_only
    if sku_name_processed_7 in master_list:
        return sku_name_processed_7
    else:
        return orig_sku_name+"- "+orig_pack_size
    
    
    
nan_final_result['Last SKU Name'] = nan_final_result.apply(lambda x :check_packsize_match(x,master_list),axis=1)
known_final_result['Last SKU Name'] = known_final_result['Final SKU Name']

others_df['second part'] = others_df['Final SKU Name'].apply(lambda x:'-'.join(x.split('-')[1:]))
others_df['Packsize'] = others_df['second part'].apply(lambda x:x.split('-')[1] if '-' in x else '')
others_df['Last SKU Name']  = others_df[['Brand Name','Product','Packsize']].apply(lambda x:str(x[0]).strip() + ' - '+str(x[1]).strip() + ' - '+x[2] if x[2]!='' else str(x[0]).strip() + ' - '+str(x[1]).strip(),axis=1)
others_df.drop(columns = ['second part','Packsize'])

ps_matched_df = pd.concat([nan_final_result,known_final_result,others_df])

ps_matched_df.to_excel("Refer Doc.xlsx",index=False)

data_file_df = ps_matched_df[['S. No.','City','Outlet Code','Outlet Name','Fin_Price','Fin_Qty','Fin_Amt','Brand Name','Product','Last SKU Name']]  # add outlet code, outlet name and city back to this line

data_file_df.to_excel("Data File.xlsx",index=False)